# L25: Intro to GPT2

Tasks:
1. Set up imports
2. Load the dataset (in a way transformers expects)
3. Define the model and tokenizer we'll be using
4. Define a function to tokenize a given name
5. Apply the tokenizer to all the names
6. Define the parameters for finetuning the model
7. Run the training loop
8. Define the parameters for generating names
9. Generate names

### Set up imports

In [2]:
# !pip install transformers datasets accelerate

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, AutoModelForCausalLM
from transformers import Trainer, TrainingArguments, pipeline

### Load the names dataset

In [9]:
dataset = load_dataset("text", data_files='names.txt')

# look at first name
dataset['train'][0]

{'text': 'emma'}

### Define the model and tokenizer

In [26]:
model_name = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name, pad_token=' ')

### Define a function to tokenize a name

In [27]:
def tokenize_name(name):
    tokens = tokenizer(name['text'] + tokenizer.eos_token)

    return tokens

# test the function
tokenized_name = tokenize_name({'text': 'garrett'})

# get the token id list
token_id_list = tokenized_name['input_ids']

# convert to tokens
token_list = tokenizer.convert_ids_to_tokens(token_id_list)

token_list

['gar', 'rett', '<|endoftext|>']

### Tokenize all the names in the dataset

In [28]:
tokenized_names = dataset['train'].map(
    tokenize_name,
    batched=False,
    remove_columns=['text']
)

tokenized_names

Map: 100%|██████████| 32033/32033 [00:03<00:00, 8126.74 examples/s]


Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 32033
})

In [29]:
# look at first tokenized name
token_id_list = tokenized_names['input_ids'][10]

token_list = tokenizer.convert_ids_to_tokens(token_id_list)

token_list

['ab', 'ig', 'ail', '<|endoftext|>']

### Prepare data from model training

In [30]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

data_collator

DataCollatorForLanguageModeling(tokenizer=GPT2Tokenizer(name_or_path='distilgpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': ' '}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	50257: AddedToken(" ", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), mlm=False, whole_word_mask=False, mlm_probability=0.15, mask_replace_prob=0.8, random_replace_prob=0.1, pad_to_multiple_of=None, return_tensors='pt', seed=None)

In [31]:
# set up the training parameters
model = AutoModelForCausalLM.from_pretrained(model_name)

# adjust the models vocab
model.resize_token_embeddings(len(tokenizer))

model

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 15297.39it/s]
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50258, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [32]:
# define how the training is done
training_parameters = TrainingArguments(
    output_dir='name_gpt',
    per_device_train_batch_size=8,
    max_steps=100,
)

### Train the model

In [33]:
trainer = Trainer(
    model=model,
    args=training_parameters,
    train_dataset=tokenized_names,
    data_collator=data_collator
)

# run the training
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.92it/s]


TrainOutput(global_step=100, training_loss=3.3449822998046876, metrics={'train_runtime': 90.6307, 'train_samples_per_second': 8.827, 'train_steps_per_second': 1.103, 'total_flos': 926786912256.0, 'train_loss': 3.3449822998046876, 'epoch': 0.024968789013732832})

In [34]:
# save the trained model
trainer.save_model('name-gpt')
tokenizer.save_pretrained('name-gpt')

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.08s/it]


('name-gpt/tokenizer_config.json', 'name-gpt/tokenizer.json')

### Use model to generate new names

In [36]:
generator = pipeline(
    'text-generation',
    model='name-gpt',
    tokenizer=tokenizer
)

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 558.88it/s]


In [39]:
generated_names = generator(
    'ki',
    do_sample=True,
    top_k=25,
    top_p=0.95,
    num_return_sequences=10,
)

for name in generated_names:
    clean_name = name['generated_text']

    print(clean_name)

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


kiyah
kian
kiyas
kiakari
kian
kiak
kiah
kiayle
kian
kiis


# Activity 1: Change the parameters of training
Try changing the batch size or the max steps. How does that affect training time and prediction quality?

# Activity 2: Change the parameters of generating 
Try turning off do_sample, or different values of top_k and top_p. How does that affect the generated names?